In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import mannwhitneyu, ttest_ind
from statsmodels.stats.proportion import proportions_ztest, proportion_effectsize
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.multitest import multipletests
import pymc as pm
import arviz as az

In [ ]:
control=pd.read_csv('control_group.csv')
test=pd.read_csv('test_group.csv')

In [ ]:
control.head()

In [ ]:
columns_list = control.columns[0].split(';')
data = list(control["Campaign Name;Date;Spend [USD];# of Impressions;Reach;# of Website Clicks;# of Searches;# of View Content;# of Add to Cart;# of Purchase"].apply(lambda x: x.split(';')))

In [ ]:
control_group = pd.DataFrame(data, columns=columns_list)
control_group['Date'] = pd.to_datetime(control_group['Date'],format='%d.%m.%Y')
control_group.head()

In [ ]:
control_group.describe()

In [ ]:
print(f"Number of rows={control_group.shape[0]}")
print(f"Number of columns={control_group.shape[1]}")

In [ ]:
print(f"Number of missing values={control_group.isna().sum()}")
print(f"Number of duplicates={control_group.duplicated().sum()}")

In [ ]:
test.head()

In [ ]:
columns_list2=test.columns[0].split(';')
data2 = list(test["Campaign Name;Date;Spend [USD];# of Impressions;Reach;# of Website Clicks;# of Searches;# of View Content;# of Add to Cart;# of Purchase"].apply(lambda x: x.split(';')))

In [ ]:
test_group = pd.DataFrame(data2, columns=columns_list2)
test_group['Date'] = pd.to_datetime(test_group['Date'],format='%d.%m.%Y')
test_group.head()

In [ ]:
test_group.describe()

In [ ]:
print(f"Number of rows={test_group.shape[0]}")
print(f"Number of columns={test_group.shape[1]}")

In [ ]:
print(f"Number of missing values={test_group.isna().sum()}")
print(f"Number of duplicates={test_group.duplicated().sum()}")

In [ ]:
print(control_group.groupby('Campaign Name')['Date'].agg(['min', 'max']))
print(test_group.groupby('Campaign Name')['Date'].agg(['min', 'max']))

In [ ]:
print(control_group.dtypes)

In [ ]:
cols_to_convert = ['Spend [USD]', '# of Impressions', 'Reach', '# of Website Clicks',
                   '# of Searches', '# of View Content', '# of Add to Cart', '# of Purchase']

for col in cols_to_convert:
    control_group[col] = pd.to_numeric(control_group[col], errors='coerce')
    test_group[col]    = pd.to_numeric(test_group[col], errors='coerce')

print(control_group.dtypes)

In [ ]:
print(f"Control total spend: ${control_group['Spend [USD]'].sum():,.2f}")
print(f"Test total spend:    ${test_group['Spend [USD]'].sum():,.2f}")

In [ ]:
df= pd.concat([control_group,test_group])

In [ ]:
spend_col = 'Spend [USD]'
funnel_cols = ['# of Impressions', 'Reach', '# of Website Clicks','# of Searches',
               '# of View Content', '# of Add to Cart', '# of Purchase']
for col in funnel_cols:
    df[f'{col}_per_1k_spend'] = np.where(
        df['Spend [USD]'] > 0,                          #only dividing if spend>0
        (df[col]/df['Spend [USD]'])*1000,
        np.nan                                           #else mark as NaN
    )
print('Spend-normalised columns created')
df[['Campaign Name'] + [f'{c}_per_1k_spend' for c in funnel_cols]].groupby('Campaign Name').mean().round(2)

In [ ]:
#Conversion funnel rates
funnel_agg = df.groupby('Campaign Name')[funnel_cols].sum()

# Conversion rates at each stage relative to impressions
funnel_rates = funnel_agg.div(funnel_agg['# of Impressions'], axis=0) * 100
print('Funnel Conversion Rates (% of Impressions)')
print(funnel_rates.round(3))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

#Plot 1: Funnel bar chart
funnel_rates[funnel_cols].T.plot(kind='bar', ax=axes[0], colormap='Set2', width=0.7)
axes[0].set_title('Conversion Funnel: % of Impressions', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Conversion Rate (%)')
axes[0].set_xlabel('')
axes[0].tick_params(axis='x', rotation=30)
axes[0].legend(title='Group')

#Plot 2: Daily purchases over time
for grp, grp_df in df.groupby('Campaign Name'):
    axes[1].plot(grp_df['Date'], grp_df['# of Purchase'], marker='o', label=grp, linewidth=2)
axes[1].set_title('Daily Purchases Over Time', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Purchases')
axes[1].set_xlabel('Date')
axes[1].legend(title='Group')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.savefig('funnel_eda.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
#Stage-to-stage drop-off rates
stages = ['# of Impressions', '# of Website Clicks','# of Searches','# of View Content', '# of Add to Cart', '# of Purchase']
dropoff = {}
for grp, gdf in df.groupby('Campaign Name'):
    totals = gdf[stages].sum()
    rates  = [totals[stages[i+1]] / totals[stages[i]] * 100 for i in range(len(stages)-1)]
    dropoff[grp] = rates

dropoff_df = pd.DataFrame(dropoff,
    index=[f'{stages[i]}→{stages[i+1]}' for i in range(len(stages)-1)])
print('Stage-to-Stage Conversion Rates(%)')
print(dropoff_df.round(2))

In [ ]:
# Re-create control_df and test_df AFTER normalisation columns exist in df
control_df = df[df['Campaign Name'] == 'Control Campaign'].copy()
test_df    = df[df['Campaign Name'] == 'Test Campaign'].copy()

# Verify normalised columns are present
print(control_df['# of Purchase_per_1k_spend'].describe().round(2))
print(test_df['# of Purchase_per_1k_spend'].describe().round(2))

In [ ]:
#Using Purchase as the primary metric(bottom-of-funnel, highest business value)
#We are treating daily purchase counts as our observations (n=30 per group)

#Primary metric: Purchase-per-1k-spend
baseline_mean = control_df['# of Purchase_per_1k_spend'].mean()
baseline_std  = control_df['# of Purchase_per_1k_spend'].std()

print(f'Control baseline (purchases per $1k spend): {baseline_mean:.2f} ± {baseline_std:.2f}')

#Minimum detectable effect:10% relative lift(business threshold)
MDE = 0.10*baseline_mean
effect_size_d = MDE/baseline_std   #Cohen's d

print(f'Minimum Detectable Effect (10% lift): {MDE:.2f}')
print(f'Cohen\'s d: {effect_size_d:.3f}')

#Required sample size
power_analysis = NormalIndPower()
required_n = power_analysis.solve_power(
    effect_size=effect_size_d,
    alpha=0.05,
    power=0.80,
    alternative='two-sided'
)

actual_n = len(control_df)
print(f'\nRequired n per group (80% power, α=0.05): {int(np.ceil(required_n))}')
print(f'Actual n per group: {actual_n}')

if actual_n >= required_n:
    print('Experiment is adequately powered.')
else:
    print(f'Experiment is UNDERPOWERED. Results should be interpreted with caution.')
    print(f'Actual power≈{power_analysis.solve_power(effect_size=effect_size_d, nobs1=actual_n, alpha=0.05):.2%}')

In [ ]:
df.head()

In [ ]:
from scipy.stats import shapiro
results = []
metrics = [
    '# of Impressions_per_1k_spend',
    '# of Website Clicks_per_1k_spend',
    '# of Searches_per_1k_spend',
    '# of View Content_per_1k_spend',
    '# of Add to Cart_per_1k_spend',
    '# of Purchase_per_1k_spend'
]

for metric in metrics:
    c = control_df[metric].dropna()
    t = test_df[metric].dropna()

    #Normality check (Shapiro-Wilk)
    _, p_norm_c = shapiro(c)
    _, p_norm_t = shapiro(t)
    is_normal = (p_norm_c > 0.05) and (p_norm_t > 0.05)

    #Choosing test based on normality
    if is_normal:
        stat, p_val = ttest_ind(c, t, equal_var=False)   # Welch's t-test
        test_used = "Welch's t-test"
    else:
        stat, p_val = mannwhitneyu(c, t, alternative='two-sided')
        test_used = 'Mann-Whitney U'

    #Effect size: Cohen's d
    pooled_std = np.sqrt((c.std()**2 + t.std()**2) / 2)
    cohens_d   = (t.mean() - c.mean()) / pooled_std if pooled_std > 0 else 0

    results.append({
        'Metric': metric.replace('_per_1k_spend', '').replace('# of ', '').title(),
        'Control Mean': round(c.mean(), 2),
        'Test Mean':    round(t.mean(), 2),
        'Lift (%)':     round((t.mean() - c.mean()) / c.mean() * 100, 1),
        'Test Used':    test_used,
        'p-value':      round(p_val, 4),
        "Cohen's d":    round(cohens_d, 3),
        'Significant (α=0.05)': 'Yes' if p_val < 0.05 else 'No'
    })

results_df = pd.DataFrame(results)
print('Frequentist Test Results (raw p-values)')
results_df

In [ ]:
p_values = results_df['p-value'].values

#Bonferroni correction
reject_bonf, p_bonf, _, _ = multipletests(p_values, alpha=0.05, method='bonferroni')

#Benjamini-Hochberg (FDR) correction
reject_bh, p_bh, _, _ = multipletests(p_values, alpha=0.05, method='fdr_bh')

results_df['p(Bonferroni)'] = p_bonf.round(4)
results_df['Sig.(Bonferroni)'] = ['Yes' if r else 'No' for r in reject_bonf]
results_df['p(BH/FDR)'] = p_bh.round(4)
results_df['Sig.(BH)'] = ['Yes' if r else 'No' for r in reject_bh]

print('Results After Multiple Comparisons Correction')
results_df[['Metric', 'Lift (%)', 'p-value', 'p(Bonferroni)',
            'Sig.(Bonferroni)', 'p(BH/FDR)', 'Sig.(BH)']]

In [ ]:
#Visualizing p-value comparison across corrections
fig, ax = plt.subplots(figsize=(11, 5))

x=np.arange(len(results_df))
w=0.25

ax.bar(x-w, results_df['p-value'],width=w, label='Raw p-value',color='#4C9BE8')
ax.bar(x,results_df['p(Bonferroni)'],width=w, label='Bonferroni',color='#E87B4C')
ax.bar(x+w, results_df['p(BH/FDR)'], width=w, label='BH/FDR',color='#6EC98F')

ax.axhline(0.05, color='red', linestyle='--', linewidth=1.2, label='α=0.05')
ax.set_xticks(x)
ax.set_xticklabels(results_df['Metric'], rotation=25, ha='right')
ax.set_ylabel('p-value')
ax.set_title('Multiple Comparisons: Raw vs Corrected p-values', fontsize=13, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('multiple_comparisons.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
#Primary metric: raw purchase counts per day
purchases_control=control_df['# of Purchase'].values
purchases_test=test_df['# of Purchase'].values

with pm.Model() as purchase_model:
    #Weakly informative Gamma priors on daily purchase rate
    lambda_control=pm.Gamma('lambda_control',alpha=2,beta=0.01)
    lambda_test=pm.Gamma('lambda_test',alpha=2,beta=0.01)

    #Likelihood: daily purchases follow a Poisson distribution
    obs_control=pm.Poisson('obs_control',mu=lambda_control,observed=purchases_control)
    obs_test=pm.Poisson('obs_test',mu=lambda_test,observed=purchases_test)

    #Derived quantities
    delta=pm.Deterministic('delta',lambda_test-lambda_control)
    rel_uplift=pm.Deterministic('rel_uplift',(lambda_test-lambda_control)/lambda_control)
    trace = pm.sample(4000, tune=2000, target_accept=0.95, return_inferencedata=True, progressbar=True)
print('Sampling complete')

In [ ]:
#Key posterior quantities
delta_samples=trace.posterior['delta'].values.flatten()
uplift_samples=trace.posterior['rel_uplift'].values.flatten()

prob_test_better=(delta_samples>0).mean()
median_uplift=np.median(uplift_samples)*100
ci_low, ci_high=np.percentile(uplift_samples*100,[2.5,97.5])

print(f'P(Test campaign > Control):{prob_test_better:.1%}')
print(f'Median relative uplift:{median_uplift:.1f}%')
print(f'95% Credible Interval for uplift: [{ci_low:.1f}%, {ci_high:.1f}%]')
print('Interpretation:')
print(f'There is a {prob_test_better:.0%} posterior probability that the test campaign generates more purchases than the control campaign, with an estimated')
print(f'median uplift of {median_uplift:.1f}% (95% CI: {ci_low:.1f}% to {ci_high:.1f}%).')

In [ ]:
#Posterior distribution plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Posterior of delta (absolute difference)
axes[0].hist(delta_samples, bins=80, color='#4C9BE8', alpha=0.8, density=True, edgecolor='white', linewidth=0.3)
axes[0].axvline(0, color='red', linestyle='--', linewidth=1.5, label='No difference')
axes[0].axvline(np.median(delta_samples), color='navy', linestyle='-', linewidth=1.5, label=f'Median = {np.median(delta_samples):.1f}')
axes[0].fill_betweenx(
    [0, axes[0].get_ylim()[1] if axes[0].get_ylim()[1]>0 else 1],
    np.percentile(delta_samples, 2.5),
    np.percentile(delta_samples, 97.5),
    alpha=0.15, color='navy', label='95% Credible Interval'
)
axes[0].set_title('Posterior: Absolute Difference in Daily Purchases\n(Test−Control)', fontweight='bold')
axes[0].set_xlabel('Δ Daily Purchases')
axes[0].set_ylabel('Density')
axes[0].legend(fontsize=9)

# Plot 2: Posterior of relative uplift
axes[1].hist(uplift_samples * 100, bins=80, color='#6EC98F', alpha=0.8, density=True, edgecolor='white', linewidth=0.3)
axes[1].axvline(0, color='red', linestyle='--', linewidth=1.5, label='No uplift')
axes[1].axvline(median_uplift, color='darkgreen', linestyle='-', linewidth=1.5, label=f'Median = {median_uplift:.1f}%')
axes[1].set_title(f'Posterior: Relative Uplift (%)\nP(Test>Control) = {prob_test_better:.1%}', fontweight='bold')
axes[1].set_xlabel('Relative Uplift (%)')
axes[1].set_ylabel('Density')
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.savefig('bayesian_posterior.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
#ArviZ summary for full posterior diagnostics
az.plot_posterior(
    trace,
    var_names=['lambda_control', 'lambda_test', 'delta', 'rel_uplift'],
    hdi_prob=0.95
)
plt.suptitle('Posterior Distributions — Full Summary', fontsize=12, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('arviz_posterior_summary.png', dpi=150, bbox_inches='tight')
plt.show()